In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from statsmodels.stats.stattools import durbin_watson

In [ ]:
# ==============================================================================
# 1. PARÁMETROS Y CONSTANTES
# ==============================================================================
L = 0.257          # Longitud de la barra (m)
x_sensor = np.array([0.05, 0.10, 0.15])  # Posición de los sensores (m)
sigma_T = 0.1      # Precisión del sensor (°C)
T_A = 15           # Temperatura ambiente (°C)

# Propiedades teóricas del hierro
k_m = 80.0         # W / m K (conductividad térmica de la barra)
rho_m = 7870.0     # kg / m^3 (densidad)
Cp_m = 460.0       # J / kg K (capacidad calorífica a presión constante)
alfa_teorico = k_m / (rho_m * Cp_m)

# Pérdidas laterales teóricas
r_int, r_ext = 0.00582, 0.0125 #(radio interno y externo de la barra)
k_A = 0.04 #(conductividad térmica del aislante)
h_conv, h_rad = 8.49, 1.35 #(coeficientes de transferencia de calor por convección y radiación)
A_m = np.pi * r_int**2 #(área de sección transversal)

R_cond = np.log(r_ext / r_int) / (2 * np.pi * k_A) #(resistencia térmica de conducción)
R_paralelo = 1.0 / ((h_conv + h_rad) * 2 * np.pi * r_ext) #(resistencia térmica de convección y radiación)
R_total_estrella = R_cond + R_paralelo
beta_teorico = 1.0 / (rho_m * Cp_m * A_m * R_total_estrella)

k1 = np.pi / (2 * L)
k2 = 3 * np.pi / (2 * L)

# ==============================================================================
# 2. CARGA DE DATOS
# ==============================================================================
datos_limpios = []
try:
    with open('prueba.csv', 'r', encoding='utf-8', errors='ignore') as f:
        for linea in f:
            linea = linea.strip()
            if not linea or linea.startswith('#'):
                continue
            partes = [p.strip() for p in linea.replace(';', ',').split(',')]
            if len(partes) >= 3:
                try:
                    t = float(partes[0])
                    pos = float(partes[1])
                    temp = float(partes[2])
                    datos_limpios.append([t, pos, temp])
                except ValueError:
                    continue
except FileNotFoundError:
    raise FileNotFoundError("No se encontró el archivo en la carpeta del proyecto.")

df = pd.DataFrame(datos_limpios, columns=['tiempo', 'posicion', 'temperatura'])

if df.empty:
    raise ValueError("El DataFrame quedó vacío después de limpiar el CSV. Revisar el contenido del archivo cargado.")

# Conversión si las medidas de posición están en centímetros
if df['posicion'].max() > 1.0:
    df['posicion'] = df['posicion'] / 100.0

# Elimina duplicados si el microcontrolador reinició registros
df = df.drop_duplicates(subset=['tiempo', 'posicion'], keep='last')

# Pivoteo de la tabla(asignando el tiempo a las filas y la posición a las columnas)
df_pivot = df.pivot_table(index='tiempo', columns='posicion', values='temperatura').reset_index()

t_c = df_pivot['tiempo'].values.astype(float)

# Detección de las columnas de los sensores
cols_disponibles = [float(c) for c in df_pivot.columns if c != 'tiempo']
if len(cols_disponibles) < 3:
    raise ValueError(f"Se esperaban al menos 3 posiciones de sensores, pero se encontraron: {cols_disponibles}")

pos_1 = min(cols_disponibles, key=lambda x: abs(x - 0.05))
pos_2 = min(cols_disponibles, key=lambda x: abs(x - 0.10))
pos_3 = min(cols_disponibles, key=lambda x: abs(x - 0.15))

# Matriz u(x_i, t_c) = T_medida - T_A
U_medido = df_pivot[[pos_1, pos_2, pos_3]].values - T_A

# ==============================================================================
# 3. MÍNIMOS CUADRADOS PARA D_1(t_c) Y D_2(t_c)
# ==============================================================================
M = np.array([
    [np.sin(k1 * 0.05), np.sin(k2 * 0.05)],
    [np.sin(k1 * 0.10), np.sin(k2 * 0.10)],
    [np.sin(k1 * 0.15), np.sin(k2 * 0.15)]
])

M_pseudo_inv = np.linalg.pinv(M)

D_matrix = np.zeros((len(t_c), 2))
for i in range(len(t_c)):
    D_matrix[i, :] = M_pseudo_inv @ U_medido[i, :]

D1, D2 = D_matrix[:, 0], D_matrix[:, 1]

# ==============================================================================
# 4. BÚSQUEDA DE VENTANA COMÚN
# ==============================================================================
piso_ruido = 2 * sigma_T
mask = (np.abs(D1) > piso_ruido) & (np.abs(D2) > piso_ruido)

t_sub = t_c[mask]
ln_D1_sub = np.log(np.abs(D1[mask]))
ln_D2_sub = np.log(np.abs(D2[mask]))

mejor_r2_conjunto = -1
idx_i_opt, idx_j_opt = 0, len(t_sub)

min_puntos = 10
for i in range(0, len(t_sub) - min_puntos, 2):
    for j in range(i + min_puntos, len(t_sub), 2):
        res1 = linregress(t_sub[i:j], ln_D1_sub[i:j])
        res2 = linregress(t_sub[i:j], ln_D2_sub[i:j])
        
        #Decaimiento físico estricto (pendientes negativas en ambos)
        if res1.slope < 0 and res2.slope < 0:
            r2_prom = (res1.rvalue**2 + res2.rvalue**2) / 2.0
            if r2_prom > mejor_r2_conjunto:
                mejor_r2_conjunto = r2_prom
                idx_i_opt, idx_j_opt = i, j

# Ventana compartida estrictamente por ambos modos
t_v = t_sub[idx_i_opt:idx_j_opt]
ln_D1_v = ln_D1_sub[idx_i_opt:idx_j_opt]
ln_D2_v = ln_D2_sub[idx_i_opt:idx_j_opt]

# ==============================================================================
# 5. REGRESIONES, ORDENAMIENTO FÍSICO Y ERRORES
# ==============================================================================
res_1 = linregress(t_v, ln_D1_v)
g_a = -res_1.slope
sigma_ga = res_1.stderr
residuos_1 = ln_D1_v - (res_1.intercept + res_1.slope * t_v)
dw_1 = durbin_watson(residuos_1)

res_2 = linregress(t_v, ln_D2_v)
g_b = -res_2.slope
sigma_gb = res_2.stderr
residuos_2 = ln_D2_v - (res_2.intercept + res_2.slope * t_v)
dw_2 = durbin_watson(residuos_2)

# Ordenamiento físico: gamma_2 > gamma_1 (el modo superior decae más rápido)
if g_a < g_b:
    gamma_1, sigma_gamma1, r2_1, dw_1_out, res_1_obj, ln_val_1, residuos_g1 = g_a, sigma_ga, res_1.rvalue**2, dw_1, res_1, ln_D1_v, residuos_1
    gamma_2, sigma_gamma2, r2_2, dw_2_out, res_2_obj, ln_val_2, residuos_g2 = g_b, sigma_gb, res_2.rvalue**2, dw_2, res_2, ln_D2_v, residuos_2
else:
    gamma_1, sigma_gamma1, r2_1, dw_1_out, res_1_obj, ln_val_1, residuos_g1 = g_b, sigma_gb, res_2.rvalue**2, dw_2, res_2, ln_D2_v, residuos_2
    gamma_2, sigma_gamma2, r2_2, dw_2_out, res_2_obj, ln_val_2, residuos_g2 = g_a, sigma_ga, res_1.rvalue**2, dw_1, res_1, ln_D1_v, residuos_1

delta_k = k2**2 - k1**2
alfa_exp = (gamma_2 - gamma_1) / delta_k
beta_exp = gamma_1 - alfa_exp * (k1**2)

# Propagación de Errores
sigma_alfa = np.sqrt(sigma_gamma1**2 + sigma_gamma2**2) / delta_k
term1 = (1.0 - (k1**2 / delta_k)) * sigma_gamma1
term2 = (k1**2 / delta_k) * sigma_gamma2
sigma_beta = np.sqrt(term1**2 + term2**2)

# ==============================================================================
# 6. RESULTADOS Y GRÁFICOS
# ==============================================================================
print(f"Temperatura Ambiente T_A: {T_A} °C")
print(f"Pendiente Modo 1 (gamma_1): {gamma_1:.6e} ± {sigma_gamma1:.6e} s^-1 (R^2 = {r2_1:.4f})")
print(f"Estadístico Durbin-Watson (Modo 1): {dw_1_out:.3f}")
print(f"Pendiente Modo 2 (gamma_2): {gamma_2:.6e} ± {sigma_gamma2:.6e} s^-1 (R^2 = {r2_2:.4f})")
print(f"Estadístico Durbin-Watson (Modo 2): {dw_2_out:.3f}")
print("----------------------------------------------------------------------")
print(f"Difusividad térmica (Alpha Exp):  {alfa_exp:.4e} ± {sigma_alfa:.4e} m^2/s")
print(f"Difusividad térmica (Alpha Teor): {alfa_teorico:.4e} m^2/s")
print("----------------------------------------------------------------------")
print(f"Coef. Pérdida Lateral (Beta Exp):  {beta_exp:.4e} ± {sigma_beta:.4e} s^-1")
print(f"Coef. Pérdida Lateral (Beta Teor): {beta_teorico:.4e} s^-1")
print("======================================================================\n")

fig, axs = plt.subplots(1, 3, figsize=(14, 5))

axs[0].plot(t_v, ln_val_1, 'bo', markersize=3, label='ln|D1|')
axs[0].plot(t_v, res_1_obj.intercept + res_1_obj.slope * t_v, 'r-', label=f'gamma_1={gamma_1:.2e}')
axs[0].plot(t_v, ln_val_2, 'go', markersize=3, label='ln|D2|')
axs[0].plot(t_v, res_2_obj.intercept + res_2_obj.slope * t_v, 'm-', label=f'gamma_2={gamma_2:.2e}')
axs[0].set_xlabel('Tiempo $t_c$ (s)')
axs[0].set_ylabel('$\ln|D_n(t_c)|$')
axs[0].set_title('Decaimiento Modal Conjunto')
axs[0].grid(True)
axs[0].legend()

axs[1].scatter(t_v, residuos_g1, color='purple', s=10)
axs[1].axhline(0, color='black', linestyle='--')
axs[1].set_xlabel('Tiempo $t_c$ (s)')
axs[1].set_ylabel('Residuos')
axs[1].set_title(f'Residuos Modo 1 (DW: {dw_1_out:.2f})')
axs[1].grid(True)

axs[2].scatter(t_v, residuos_g2, color='purple', s=10)
axs[2].axhline(0, color='black', linestyle='--')
axs[2].set_xlabel('Tiempo $t_c$ (s)')
axs[2].set_ylabel('Residuos')
axs[2].set_title(f'Residuos Modo 2 (DW: {dw_2_out:.2f})')
axs[2].grid(True)

plt.tight_layout()
plt.show()